In [0]:
csv_path = "/Workspace/Practise/Sample - Superstore.csv" 

# Read CSV with quote escaping so it doesn't break on commas inside product names
df_initial = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .load(csv_path)

# Clean duplicates
df_cleaned = df_initial.dropDuplicates(["Row ID"]).dropna(subset=["Row ID", "Customer ID"])

# Sanitize all column names to remove spaces and hyphens
safe_columns = [c.replace(" ", "_").replace("-", "_") for c in df_cleaned.columns]
df_cleaned = df_cleaned.toDF(*safe_columns)

catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
schema_name = spark.sql("SELECT current_database()").collect()[0][0]
table_name = f"{catalog_name}.{schema_name}.superstore_master"

# THE FIX: Drop the old broken table entirely so Delta accepts our new schema!
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Save fresh Managed Table
df_cleaned.write.format("delta").mode("overwrite").saveAsTable(table_name)
print(f"Fresh Delta table created with perfect schema at: {table_name}")

Fresh Delta table created with perfect schema at: workspace.default.superstore_master


In [0]:
from pyspark.sql.functions import col, lit

catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
schema_name = spark.sql("SELECT current_database()").collect()[0][0]
table_name = f"{catalog_name}.{schema_name}.superstore_master"

# Fetch 5 existing rows to simulate updates
existing_rows = spark.table(table_name).limit(5)
updated_batch = existing_rows.withColumn("Sales", col("Sales") * 1.10) \
                             .withColumn("Profit", col("Profit") * 1.05)

# Create 1 completely new row (Insert) using the cleaned schema
new_row_template = spark.table(table_name).limit(1)
new_batch = new_row_template.withColumn("Row_ID", lit(99999)) \
                            .withColumn("Order_ID", lit("CA-2026-99999")) \
                            .withColumn("Customer_Name", lit("Nilesh Sharma")) \
                            .withColumn("Sales", lit(550.00)) \
                            .withColumn("Profit", lit(120.00))

# Combine into one incremental dataframe
df_incremental = updated_batch.union(new_batch)
df_incremental.createOrReplaceTempView("incremental_batch")

print(f"Incremental batch ready: {df_incremental.count()} rows (5 updates, 1 insert).")

Incremental batch ready: 6 rows (5 updates, 1 insert).


In [0]:
from delta.tables import DeltaTable

catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
schema_name = spark.sql("SELECT current_database()").collect()[0][0]
table_name = f"{catalog_name}.{schema_name}.superstore_master"

# Connect to the live Delta table
master_delta = DeltaTable.forName(spark, table_name)

# Execute the Upsert logic
master_delta.alias("target") \
  .merge(
    df_incremental.alias("source"),
    "target.Row_ID = source.Row_ID"
  ) \
  .whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()

print("Delta Merge processing successfully completed.")

Delta Merge processing successfully completed.


In [0]:
from pyspark.sql.functions import col
# Ensure we have our dynamic table name
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
schema_name = spark.sql("SELECT current_database()").collect()[0][0]
table_name = f"{catalog_name}.{schema_name}.superstore_master"

# Fetch the freshly updated table
df_final = spark.table(table_name)

# 1. Validate Row Count (Should be exactly 1 higher than your initial load)
print(f"Final Table Row Count: {df_final.count()}")

# 2. Duplicate Check (Should be 0, proving the MERGE updated instead of duplicating)
duplicate_count = df_final.groupBy("Row_ID").count().filter("count > 1").count()
print(f"Duplicate records found on business key: {duplicate_count}")

# 3. Prove the insert worked by querying your specific row
print("Showing the newly inserted record:")
df_final.filter(col("Row_ID") == 99999).show()

Final Table Row Count: 9995
Duplicate records found on business key: 0
Showing the newly inserted record:
+------+-------------+----------+----------+--------------+-----------+-------------+--------+-------------+-----------+----------+-----------+------+---------------+----------+------------+--------------------+-----+--------+--------+------+
|Row_ID|     Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|Customer_Name| Segment|      Country|       City|     State|Postal_Code|Region|     Product_ID|  Category|Sub_Category|        Product_Name|Sales|Quantity|Discount|Profit|
+------+-------------+----------+----------+--------------+-----------+-------------+--------+-------------+-----------+----------+-----------+------+---------------+----------+------------+--------------------+-----+--------+--------+------+
| 99999|CA-2026-99999|2014-06-09|2014-06-14|Standard Class|   BH-11710|Nilesh Sharma|Consumer|United States|Los Angeles|California|      90032|  West|TEC-PH-10002033

In [0]:
#The transaction history of the table:
display(spark.sql(f"DESCRIBE HISTORY {table_name}"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-07-12T21:22:34.000Z,70381650098820,sharmanilesh7105@gmail.com,MERGE,"Map(predicate -> [""(Row_ID#15284 = Row_ID#15128)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3096654546232155),d33fed2f-5326-4dc4-b5f5-007d0381a355,0712-205905-or3x2f6d-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 12284, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 8063, materializeSourceTimeMs -> 836, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3347, numTargetRowsUpdated -> 5, numOutputRows -> 6, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 6, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3570)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-12T21:20:31.000Z,70381650098820,sharmanilesh7105@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3096654546232155),7bf387b5-db5c-487c-b942-6982e0347f67,0712-205905-or3x2f6d-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 9994, numOutputBytes -> 332983)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
